---

## ToolRetryMiddleware 옵션별 테스트

``ToolRetryMiddleware`` 는 실패한 **도구 호출**을 지수 백오프(exponential backoff)와
지터(jitter)로 자동 재시도합니다.

- 훅: ``wrap_tool_call`` — ``handler(request)`` 실패 시 재시도

**참고:** [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

§1~§6은 LLM 없이 ``invoke_tool_with_retry`` 로 미들웨어만 검증합니다.
§7~§8은 ``create_agent`` / ``MiddlewareToolRetryAgent`` 로 에이전트를 실행합니다 (``OPENAI_API_KEY`` 필요).

**옵션 요약:**


| 옵션 | 기본값 | 쉽게 풀은 의미 | 개발 및 보안 관점의 팁 |
| --- | --- | --- | --- |
| **`max_retries`** | `3` | "실패 시 몇 번 더 찔러볼까?" (최초 1회 제외 추가 횟수) | 너무 크게 잡으면 전체 요청 처리 시간이 길어져 시스템 병목이 생길 수 있습니다. |
| **`initial_delay`** | `1.0` | "첫 실패 후 몇 초 쉬고 다시 할까?" | 외부 서비스가 잠깐 흔들렸을 때 회복할 최소한의 시간을 줍니다. |
| **`backoff_factor`** | `2.0` | "연속 실패 시 대기 시간을 몇 배로 늘릴까?" | 대기 시간이 지수형태(1초→2초→4초→8초)로 늘어나 상대 서버의 부하를 줄여줍니다. |
| **`max_delay`** | `60.0` | "아무리 많이 쉬어도 최대 몇 초까지만 쉴까?" | 대기 시간이 무한정 늘어나는 것을 막아 유저 응답 지연(Timeout)을 방어합니다. |
| **`jitter`** | `True` | "대기 시간에 랜덤(무작위) 시간 오차를 섞을까?" | 여러 요청이 동시에 재시도되어 서버를 또 터뜨리는 **Thundering Herd 현상을 방지**하는 핵심 보안/안정성 옵션입니다. |
| **`retry_on`** | `(Exception,)` | "어떤 에러가 났을 때만 재시도를 돌릴까?" | 401(인증 실패), 403(권한 없음), 400(파라미터 오류) 같은 **보안 및 규격 에러는 재시도 대상에서 제외**해야 리소스 낭비를 막습니다. |
| **`on_failure`** | `"continue"` | "지정된 재시도를 다 쓰고도 실패하면 어떡할까?" | `"error"`는 상위로 예외를 던져 멈추고, `"continue"`는 실패 메시지를 LLM에 넘겨 **우회(Fallback) 로직**을 타도록 유도합니다. |
| **`tools`** | `None`(전체대상) | "어떤 도구(API)에 이 재시도 규칙을 적용할까?" | 안정적인 내부 기능은 제외하고, 네트워크 상태가 불안정한 외부 API나 특정 AI 서비스 도구만 타겟팅할 때 씁니다. |

**실행 흐름**

```
handler 호출 → 성공 → ToolMessage 반환
            → 실패 → retry_on 확인 → 재시도 가능 & 횟수 남음 → backoff 대기 → 재시도
                              → 재시도 불가 또는 횟수 소진 → on_failure 처리
```

테스트 §1~§6은 ``make_tool_retry_middleware(..., initial_delay=0, backoff_factor=0, jitter=False)`` 로 빠르게 검증합니다.

In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver

from feature.MiddlewareToolCallLimit import (
    make_tool_call_limit_middleware,
    messages_contain_blocked_tool_call,
)
from feature.MiddlewareToolRetry import (
    MiddlewareToolRetryAgent,
    get_weather,
    invoke_tool_with_retry,
    is_tool_retry_failure_message,
    make_flaky_api_tool,
    make_tool_retry_middleware,
)
from util.chat_model_enums import LangChainChatModel

# 테스트 §1~§6: 백오프·지터 없이 즉시 재시도
_RETRY_TEST_KWARGS = {"initial_delay": 0.0, "backoff_factor": 0.0, "jitter": False}

### 1. 첫 호출 성공 — 재시도 없음

정상 도구는 ``wrap_tool_call`` 이 1회만 실행합니다.

In [2]:
mw = make_tool_retry_middleware(max_retries=3, **_RETRY_TEST_KWARGS)

msg = invoke_tool_with_retry(mw, get_weather, {"city": "Seoul"})

assert msg.status != "error"
assert "sunny" in msg.content
print("✓ 첫 호출 성공 —", msg.content)

✓ 첫 호출 성공 — It's sunny in Seoul!


### 2. 일시 실패 후 성공 — 재시도 복구

``make_flaky_api_tool(fail_count=2)`` 는 처음 2회 실패, 3번째 성공합니다.
``max_retries=3`` 이면 총 4회 시도(초기 1 + 재시도 3) 안에 복구됩니다.

In [3]:
flaky_tool, get_calls = make_flaky_api_tool(fail_count=2)
mw = make_tool_retry_middleware(max_retries=3, **_RETRY_TEST_KWARGS)

msg = invoke_tool_with_retry(mw, flaky_tool, {"city": "Seoul"})

assert "API OK" in msg.content
assert get_calls() == 3
print(f"✓ {get_calls()}회 시도 후 성공 —", msg.content)

✓ 3회 시도 후 성공 — API OK in Seoul!


### 3. 재시도 소진 — ``on_failure='continue'`` (기본)

모든 재시도가 실패하면 ``status='error'`` 인 ``ToolMessage`` 가 반환됩니다.

In [6]:
flaky_tool, get_calls = make_flaky_api_tool(fail_count=10)
mw = make_tool_retry_middleware(
    max_retries=2, on_failure="continue", **_RETRY_TEST_KWARGS
)

msg = invoke_tool_with_retry(mw, flaky_tool, {"city": "Busan"})

assert is_tool_retry_failure_message(msg)
assert get_calls() == 3  # 초기 1 + 재시도 2
print(f"✓ {get_calls()}회 시도 후 실패 메시지 —", msg.content[:80])

✓ 3회 시도 후 실패 메시지 — Tool 'flaky_api' failed after 3 attempts with ConnectionError: temporary network


### 4. ``on_failure='error'`` — 예외 재발생

재시도를 모두 소진하면 원래 예외가 다시 올라옵니다.

In [5]:
flaky_tool, _ = make_flaky_api_tool(fail_count=5)
mw = make_tool_retry_middleware(
    max_retries=1, on_failure="error", **_RETRY_TEST_KWARGS
)

try:
    invoke_tool_with_retry(mw, flaky_tool, {"city": "Daegu"})
    raise AssertionError("예외가 발생해야 합니다")
except ConnectionError as e:
    print(f"✓ on_failure=error — {type(e).__name__}: {e}")

✓ on_failure=error — ConnectionError: temporary network error


### 5. ``retry_on`` — 재시도 대상 예외 필터

``ValueError`` 는 재시도하지 않고 즉시 ``on_failure`` 처리합니다.

In [7]:
flaky_tool, get_calls = make_flaky_api_tool(
    fail_count=1,
    exception_type=ValueError,
    message="bad request",
)
mw = make_tool_retry_middleware(
    max_retries=3,
    retry_on=(ConnectionError,),  # ValueError 는 재시도 안 함
    **_RETRY_TEST_KWARGS,
)

msg = invoke_tool_with_retry(mw, flaky_tool, {"city": "Incheon"})

assert is_tool_retry_failure_message(msg)
assert get_calls() == 1  # 재시도 없이 1회만
assert "ValueError" in msg.content
print(f"✓ retry_on 필터 — {get_calls()}회,", msg.content[:70])

✓ retry_on 필터 — 1회, Tool 'flaky_api' failed after 1 attempt with ValueError: bad request. 


### 6. ``tools`` — 특정 도구만 재시도

``tools=['get_weather']`` 이면 ``flaky_api`` 는 재시도 래핑 없이 handler 1회만 실행되고 예외가 그대로 전파됩니다.

In [8]:
flaky_tool, get_calls = make_flaky_api_tool(fail_count=5)
mw = make_tool_retry_middleware(
    max_retries=3,
    tools=[get_weather],  # flaky_api 제외 → 재시도 래핑 없이 handler 1회만
    **_RETRY_TEST_KWARGS,
)

try:
    invoke_tool_with_retry(mw, flaky_tool, {"city": "Daejeon"})
    raise AssertionError("재시도 대상이 아니면 예외가 그대로 전파되어야 합니다")
except ConnectionError as e:
    assert get_calls() == 1
    print(f"✓ tools 필터 — flaky_api {get_calls()}회, 예외 전파: {e}")

✓ tools 필터 — flaky_api 1회, 예외 전파: temporary network error


### 7. 에이전트 통합 — 정상 도구 호출

``MiddlewareToolRetryAgent`` + ``get_weather`` 로 LLM이 도구를 호출하면
재시도 미들웨어가 붙은 채로 정상 응답이 나와야 합니다.

In [11]:
agent = MiddlewareToolRetryAgent(
    middleware=make_tool_retry_middleware(),
)

result = agent.invoke(
    inputs={"messages": [HumanMessage(content="What is the weather in Seoul? Use the tool.")]},
)

assert result is not None
last = result["messages"][-1].content
assert isinstance(last, str) and len(last) > 0
print("✓ 에이전트 통합 —", last[:120])


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_lQIUZQB9Tk2e1ZdiKNZpIR03)
 Call ID: call_lQIUZQB9Tk2e1ZdiKNZpIR03
  Args:
    city: Seoul

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is sunny!
✓ 에이전트 통합 — The weather in Seoul is sunny!


### 8. ``ToolCallLimitMiddleware`` 와 조합

두 미들웨어는 **서로 다른 훅**에서 동작하므로 함께 붙일 수 있습니다.

| 미들웨어 | 훅 | 역할 |
|:---|:---|:---|
| ``ToolRetryMiddleware`` | ``wrap_tool_call`` | 도구 **실행** 실패 시 재시도 |
| ``ToolCallLimitMiddleware`` | ``after_model`` | 모델이 **제안한** 도구 호출 횟수 제한 |

재시도는 같은 ``tool_call_id`` 를 다시 실행할 뿐이라, ``run_limit`` 카운트에는 **추가로 잡히지 않습니다**.
즉, 일시적 API 오류는 재시도로 복구하고, 무한·과다 호출은 ``ToolCallLimitMiddleware`` 로 막는 패턴입니다.

``middleware=[...]`` 리스트에 **둘 다** 넘깁니다 (순서는 훅이 달라 큰 영향 없음).

In [12]:
# ToolRetry(실행 재시도) + ToolCallLimit(호출 횟수 제한) 동시 적용
combo_agent = create_agent(
    model=init_chat_model(LangChainChatModel.OPENAI_GPT_4O_MINI),
    tools=[get_weather],
    middleware=[
        make_tool_retry_middleware(max_retries=2),
        make_tool_call_limit_middleware(run_limit=10),
    ],
    checkpointer=MemorySaver(),
)

result = combo_agent.invoke(
    {"messages": [HumanMessage(content="What is the weather in Seoul? Use get_weather once.")]},
    config=RunnableConfig(configurable={"thread_id": "retry-limit-ok"}),
)

assert result is not None
print("✓ 조합 — 정상 응답:", result["messages"][-1].content[:120])

✓ 조합 — 정상 응답: The weather in Seoul is sunny!


#### 8-1. ``run_limit=1`` — 여러 도구 제안 시 차단

Seoul·Tokyo·Paris 를 한 번에 물으면 모델이 ``get_weather`` 를 여러 번 부를 수 있습니다.
``run_limit=1`` 이면 **첫 호출만 실행**되고 나머지는 ``ToolCallLimitMiddleware`` 가 차단합니다.
(실행 중 일시 오류가 나면 ``ToolRetryMiddleware`` 가 같은 호출을 재시도합니다.)

In [13]:
combo_limit_agent = create_agent(
    model=init_chat_model(LangChainChatModel.OPENAI_GPT_4O_MINI),
    tools=[get_weather],
    middleware=[
        make_tool_retry_middleware(max_retries=2),
        make_tool_call_limit_middleware(
            tool_name="get_weather",
            run_limit=1,
            exit_behavior="continue",
        ),
    ],
    checkpointer=MemorySaver(),
)

result = combo_limit_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Use get_weather for Seoul, Tokyo, and Paris. Call each city separately."
            )
        ]
    },
    config=RunnableConfig(configurable={"thread_id": "retry-limit-block"}),
)

blocked = messages_contain_blocked_tool_call(result["messages"])
assert blocked
print(f"✓ 조합 + run_limit=1 — 차단 ToolMessage 존재: {blocked}")
err_tools = [
    m.content
    for m in result["messages"]
    if hasattr(m, "status") and m.status == "error"
]
if err_tools:
    print("  차단 메시지:", err_tools[0])

✓ 조합 + run_limit=1 — 차단 ToolMessage 존재: True
  차단 메시지: Tool call limit exceeded. Do not call 'get_weather' again.
